# ONIA — Similaritate cosinus și căutarea documentelor

**NLP · Lecția 5.8.** Exemple executabile de la produs scalar la un motor lexical TF-IDF, NearestNeighbors, recomandări și evaluare. Rulează celulele pe rând; fiecare exemplu își definește datele.

Instalare (dacă este necesar): `pip install numpy scikit-learn scipy joblib`. Datele și graficele sunt educaționale. Pentru un corpus real, vocabularul și IDF se învață doar din datele de antrenare/indexare. Interogările noi folosesc `transform`.

## 1. 01. Produs scalar și norme calculate manual

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
import math

def produs_scalar(a, b):
    return sum(x * y for x, y in zip(a, b))

def norma(a):
    return math.sqrt(sum(x * x for x in a))

q = [1, 0]
a = [1, 1]
print("produs scalar:", produs_scalar(q, a))
print("norma q:", norma(q))
print("norma a:", norma(a))


## 2. 02. Cosinus from scratch, inclusiv cazul vectorului nul

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
import math

def cosinus(a, b):
    if len(a) != len(b):
        raise ValueError("Vectorii au dimensiuni diferite")
    prod = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    if na == 0 or nb == 0:
        return 0.0  # convenție pentru motorul de căutare
    return prod / (na * nb)

for a in ([1, 0], [1, 1], [0, 1], [0, 0]):
    print(a, round(cosinus([1, 0], a), 3))


## 3. 03. BoW din liste, fără biblioteci externe

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
from collections import Counter

def tokenizeaza(text):
    return text.casefold().split()

documente = ["python date date", "python algoritmi", "sport tenis"]
vocabular = sorted({w for doc in documente for w in tokenizeaza(doc)})

for doc in documente:
    frecvente = Counter(tokenizeaza(doc))
    vector = [frecvente.get(termen, 0) for termen in vocabular]
    print(doc, "->", vector)
print("Vocabular:", vocabular)


## 4. 04. TF-IDF from scratch, cu formula scikit-learn

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
import math
from collections import Counter

docs = ["python date date", "python algoritmi", "sport tenis"]
tokens = [doc.split() for doc in docs]
vocab = sorted({w for doc in tokens for w in doc})
N = len(docs)
df = {w: sum(w in doc for doc in tokens) for w in vocab}
idf = {w: math.log((1 + N) / (1 + df[w])) + 1 for w in vocab}

for doc in tokens:
    count = Counter(doc)
    brut = [count.get(w, 0) * idf[w] for w in vocab]
    norma = math.sqrt(sum(x * x for x in brut))
    normalizat = [x / norma if norma else 0 for x in brut]
    print([round(x, 3) for x in normalizat])
print("IDF:", {w: round(v, 3) for w, v in idf.items()})


## 5. 05. Mini-motor de căutare de la zero

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
import math
from collections import Counter

docs = ["python date", "sport tenis", "modele python date", "algoritm python"]
query = "python date"
corpus = [doc.split() for doc in docs]
vocab = sorted({w for tokens in corpus for w in tokens})
N = len(docs)
idf = {w: math.log((1 + N) / (1 + sum(w in d for d in corpus))) + 1
       for w in vocab}

def vectorizeaza(text):
    tf = Counter(text.split())
    v = [tf.get(w, 0) * idf[w] for w in vocab]
    norm = math.sqrt(sum(x * x for x in v))
    return [x / norm if norm else 0.0 for x in v]

q = vectorizeaza(query)
scoruri = [sum(x*y for x, y in zip(q, vectorizeaza(doc))) for doc in docs]
for i in sorted(range(N), key=lambda i: (-scoruri[i], i))[:3]:
    print(i, round(scoruri[i], 3), docs[i])


## 6. 06. Similaritate cu NumPy și scikit-learn

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

a = np.array([[1., 1.], [1., 0.], [0., 0.]])
q = np.array([[1., 0.]])
print(cosine_similarity(q, a).round(3))


## 7. 07. Indexare și interogare corectă cu TfidfVectorizer

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus = ["python date", "sport tenis", "modele python pentru date"]
vectorizer = TfidfVectorizer(norm="l2", ngram_range=(1, 2))
index = vectorizer.fit_transform(corpus)

interogare = "python și date"
q = vectorizer.transform([interogare])  # fără fit_transform!
scoruri = cosine_similarity(q, index).ravel()
for i in sorted(range(len(corpus)), key=lambda i: (-scoruri[i], i)):
    print(i, round(float(scoruri[i]), 3), corpus[i])


## 8. 08. linear_kernel versus cosine_similarity

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel

corpus = ["python date", "date algoritmi", "fotbal sport"]
v = TfidfVectorizer(norm="l2")
X = v.fit_transform(corpus)
q = v.transform(["python date"])
print(np.allclose(cosine_similarity(q, X), linear_kernel(q, X)))

v2 = TfidfVectorizer(norm=None)
X2 = v2.fit_transform(corpus)
q2 = v2.transform(["python date"])
print(cosine_similarity(q2, X2).round(3))
print(linear_kernel(q2, X2).round(3))


## 9. 09. Căutare cu scor minim și rezultate top-k

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = ["python numpy pandas", "matplotlib grafice", "fotbal sport", "python scikit learn"]
v = TfidfVectorizer()
X = v.fit_transform(docs)
q = v.transform(["python date numpy"])
scoruri = cosine_similarity(q, X).ravel()
k, prag = 3, 0.05
ranking = sorted(range(len(docs)), key=lambda i: (-scoruri[i], i))
for i in [i for i in ranking if scoruri[i] >= prag][:k]:
    print(f"D{i} scor={scoruri[i]:.3f} | {docs[i]}")


## 10. 10. NearestNeighbors: transformarea distanței în similaritate

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

corpus = ["python numpy", "sport tenis", "python pandas date", "fotbal stadion"]
v = TfidfVectorizer()
X = v.fit_transform(corpus)
model = NearestNeighbors(n_neighbors=2, metric="cosine", algorithm="brute")
model.fit(X)
q = v.transform(["python date"])
distante, indici = model.kneighbors(q)
for d, i in zip(distante[0], indici[0]):
    print(i, f"cosinus={1 - d:.3f}", corpus[i])


## 11. 11. N-grame de cuvinte versus n-grame de caractere

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus = ["programare python", "progrmare în python", "analiză statistică", "python algoritmi"]
q = "programare"
configuratii = {
    "cuvinte": dict(analyzer="word", ngram_range=(1, 2)),
    "caractere": dict(analyzer="char_wb", ngram_range=(3, 5))
}
for nume, params in configuratii.items():
    v = TfidfVectorizer(**params)
    X = v.fit_transform(corpus)
    scoruri = cosine_similarity(v.transform([q]), X).ravel()
    print(nume, [(i, round(float(scoruri[i]), 3)) for i in scoruri.argsort()[::-1]])


## 12. 12. sublinear_tf și repetarea unui termen

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus = ["python " * 20 + "date", "python date date algoritmi", "sport tenis"]
for sublinear in (False, True):
    v = TfidfVectorizer(sublinear_tf=sublinear)
    X = v.fit_transform(corpus)
    q = v.transform(["python date"])
    print(sublinear, cosine_similarity(q, X).ravel().round(3))


## 13. 13. Matrice document–document și recomandări

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

docs = ["regresie logistică clasificare", "arbori clasificare", "regresie liniară", "analiză de date"]
X = TfidfVectorizer().fit_transform(docs)
S = linear_kernel(X, X)
i = 0
vecini = [j for j in range(len(docs)) if j != i]
for j in sorted(vecini, key=lambda j: -S[i, j])[:2]:
    print(f"{S[i, j]:.3f}", docs[j])


## 14. 14. Detectarea aproape-duplicatelor

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

texte = [
    "ghid python pentru începători",
    "ghid python pentru începători ediție nouă",
    "introducere în algoritmi de grafuri",
    "ghid complet python pentru începători"
]
X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit_transform(texte)
S = cosine_similarity(X)
prag = 0.60  # doar pentru demonstrație, trebuie validat
for i in range(len(texte)):
    for j in range(i + 1, len(texte)):
        if S[i, j] >= prag:
            print(i, j, round(S[i, j], 3))


## 15. 15. Relevanță: Precision@k, Recall@k și MRR

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
# fiecare listă conține ID-urile rezultatelor întoarse pentru o interogare
rezultate = [[1, 5, 3, 4], [4, 2, 0, 1], [2, 0, 1, 4]]
relevante = [{1, 3, 6}, {2}, {7}]

def precision_at_k(ranking, gold, k):
    return sum(i in gold for i in ranking[:k]) / k

def recall_at_k(ranking, gold, k):
    return sum(i in gold for i in ranking[:k]) / len(gold) if gold else 0.0

def reciprocal_rank(ranking, gold):
    return next((1 / poz for poz, i in enumerate(ranking, 1) if i in gold), 0.0)

for r, gold in zip(rezultate, relevante):
    print(round(precision_at_k(r, gold, 3), 3),
          round(recall_at_k(r, gold, 3), 3),
          round(reciprocal_rank(r, gold), 3))
print("MRR:", sum(reciprocal_rank(r, g) for r, g in zip(rezultate, relevante)) / len(rezultate))


## 16. 16. nDCG pentru relevanță gradată

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
import math

# gradele de relevanță în ordinea clasamentului: 0=irelevant, 1=util, 2=foarte util
grade = [2, 0, 1, 2, 0]

def dcg(ranking):
    return sum((2 ** rel - 1) / math.log2(i + 2)
               for i, rel in enumerate(ranking))

ideal = sorted(grade, reverse=True)
ndcg = dcg(grade) / dcg(ideal) if dcg(ideal) else 0.0
print("DCG:", round(dcg(grade), 3))
print("nDCG:", round(ndcg, 3))


## 17. 17. Salvarea indexului și ID-urilor documentelor

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
from pathlib import Path
import json
from joblib import dump, load
from sklearn.feature_extraction.text import TfidfVectorizer

out = Path("index_cautare")
out.mkdir(exist_ok=True)
iduri = ["lec-01", "lec-02", "lec-03"]
texte = ["clasificare cu svm", "regresie liniară", "rețele neurale"]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texte)
dump(vectorizer, out / "vectorizer.joblib")
dump(X, out / "index.joblib")
(out / "documente.json").write_text(
    json.dumps([{"id": i, "text": t} for i, t in zip(iduri, texte)], ensure_ascii=False),
    encoding="utf-8"
)
print(load(out / "index.joblib").shape)


## 18. 18. Căutare pe corpus cu metadate și filtrare de categorie

Rulează exemplul, verifică rezultatul și încearcă să modifici un parametru sau un document.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

date = [
    {"id": 101, "categorie": "ML", "text": "regresie logistică clasificare"},
    {"id": 102, "categorie": "NLP", "text": "clasificarea textelor cu tf idf"},
    {"id": 103, "categorie": "ML", "text": "arbori de decizie clasificare"},
    {"id": 104, "categorie": "NLP", "text": "lemmatizare și tokenizare text"}
]
v = TfidfVectorizer(ngram_range=(1, 2))
X = v.fit_transform([d["text"] for d in date])
scoruri = cosine_similarity(v.transform(["clasificare" ]), X).ravel()
rez = [i for i, d in enumerate(date) if d["categorie"] == "NLP"]
rez.sort(key=lambda i: (-scoruri[i], date[i]["id"]))
for i in rez:
    print(date[i]["id"], round(float(scoruri[i]), 3), date[i]["text"])


## Sarcini pentru experimentare

1. Adaugă documente despre alte capitole ONIA și construiește un motor top-5.
2. Compară `ngram_range=(1,1)` și `(1,2)` pe expresii cu negații.
3. Compară `analyzer="word"` și `"char_wb"` pe texte cu greșeli de tastare.
4. Calculează Precision@3 și MRR pe 5 interogări pentru care ai stabilit manual relevanța.
5. Identifică două documente despre aceeași temă care nu sunt duplicate și explică scorurile.
6. Scrie un raport de o pagină despre ce funcționează și când un sistem lexical are nevoie de completări semantice.
